# 03-1 실습 — 원시 이벤트를 분석 가능한 데이터로 바꾸기

조사 파이프라인에 방화벽 이벤트 `DENY,198.51.100.9,443,0.85,`와 연계 파일의 시작 바이트 `b"MZ"`가 함께 들어왔다고 가정합니다. 이번 실습의 임무는 각 값의 의미를 확인하고, 이후 조건문·반복문·파일 분석에서 사용할 수 있는 이벤트 프로필로 바꾸는 것입니다.

외부 패키지나 프로젝트 모듈은 사용하지 않습니다. 각 코드 셀은 **결과 예측 → 실행 → 이유 설명 → 값 변경** 순서로 학습합니다.

## 1. 임무 브리핑 — 같은 모양, 다른 의미

아래 셀을 실행하기 전에 네 줄의 출력 결과를 먼저 적어 보세요.

In [ ]:
print(10 + 20)
print("10" + "20")
print(10 == "10")
print(bool("False"))

예상과 달랐던 결과가 있다면 그 이유를 한 문장으로 적으세요. 특히 `bool()`은 문자열의 뜻이 아니라 문자열이 비어 있는지를 확인합니다. 이 차이를 놓치면 코드는 실행되면서도 탐지 여부를 반대로 해석할 수 있습니다.

## 2. 1단계 — 사건의 각 사실에 이름과 역할 부여하기

변수 이름은 사람에게 필드의 의미를 알려 주고, 자료형은 Python에 가능한 연산을 알려 줍니다. 분석 질문에 맞는 자료형으로 이벤트 한 건을 표현합니다.

In [ ]:
source_ip = "198.51.100.9"
destination_port = 443
risk_score = 0.85
is_detected = True
country = None
file_signature = b"MZ"

print(source_ip, type(source_ip))
print(destination_port, type(destination_port))
print(risk_score, type(risk_score))
print(is_detected, type(is_detected))
print(country, type(country))
print(file_signature, type(file_signature))

다음 질문에 답하세요.

1. IP는 여러 숫자가 보이는데 왜 `int`가 아닌가요?
2. `country`를 빈 문자열 대신 `None`으로 표현하면 어떤 의미가 분명해지나요?
3. `file_signature`는 왜 `str`이 아니라 `bytes`인가요?

## 3. 2단계 — 화면의 443을 그대로 믿지 않기

명령행 인자, CSV, 로그에서 읽은 포트는 화면에 숫자로 보여도 문자열일 수 있습니다. 각 출력 결과를 예상한 뒤, 어떤 값만 범위 비교와 계산에 사용할 수 있는지 설명하세요.

In [ ]:
number_port = 443
text_port = "443"

print(number_port + 1)
print(text_port + "1")
print(number_port == text_port)
print(int(text_port) + 1)

# 오류 메시지를 관찰하려면 아래 줄의 주석을 하나씩 제거하세요.
# print(text_port + 1)
# print(int("443a"))
# print(int("443.0"))

## 4. 3단계 — 원본을 분석 값으로 바꾸되 의미 지키기

형변환은 단순한 모양 변경이 아닙니다. `int(443.9)`는 반올림이 아니라 소수 부분을 제거하고, `bool("False")`는 문자열이 비어 있지 않으므로 `True`입니다. 실행 결과가 원래 데이터의 의미와 같은지 확인하세요.

In [ ]:
print(int(443.9))
print(float("443"))
print(str(443), type(str(443)))
print(bool(0))
print(bool(""))
print(bool("False"))
print(isinstance(True, bool))
print(isinstance(True, int))

## 5. 조사 결과 0과 미조사를 구분하기

네 값은 모두 프로그램에서 '없다'처럼 보일 수 있지만 의미가 다릅니다. DFIR에서 `0`은 확인 결과가 0건이라는 뜻이고, `None`은 아직 확인하지 못했다는 뜻일 수 있습니다.

In [ ]:
failed_count = 0
country = None
username = ""
is_detected = False

print(failed_count, type(failed_count))
print(country, type(country))
print(username, type(username))
print(is_detected, type(is_detected))
print(country is None)
print(failed_count == country)

## 6. 4단계 — 원본을 보존하고 이벤트 프로필 만들기

처음의 사건으로 돌아갑니다. `raw_` 변수는 수집 당시의 원본이므로 덮어쓰지 않고, 분석에 사용할 값을 별도로 만듭니다. 셀을 실행한 뒤 원본 값 하나를 변경해 결과를 다시 확인하세요.

In [ ]:
raw_ip = "198.51.100.9"
raw_port = "443"
raw_score = "0.85"
raw_action = "DENY"
raw_country = ""
file_signature = b"MZ"

port = int(raw_port)
risk_score = float(raw_score)
is_detected = raw_action == "DENY"
country = None
is_valid_port = 1 <= port <= 65535

print(raw_ip, type(raw_ip))
print(port, type(port))
print(risk_score, type(risk_score))
print(is_detected, type(is_detected))
print(country)
print(is_valid_port, type(is_valid_port))
print(file_signature, type(file_signature))

### 프로필 자기점검

아래 셀이 오류 없이 끝나면 기본 요구사항을 충족한 것입니다.

In [ ]:
assert raw_ip == "198.51.100.9"
assert port == 443
assert type(port) is int
assert risk_score == 0.85
assert type(risk_score) is float
assert is_detected is True
assert country is None
assert is_valid_port is True
assert file_signature == b"MZ"

print("모든 자기점검을 통과했습니다.")

## 7. 5단계 — 변환된 값의 신뢰 경계 확인하기

형변환 성공과 업무 범위 충족은 다른 문제입니다. `70000`은 정수로 바뀌지만 유효한 포트는 아닙니다. 아래 셀은 아직 조건문을 사용하지 않고 두 결과를 확인합니다.

In [ ]:
raw_port = "70000"
port = int(raw_port)
is_valid_range = 1 <= port <= 65535

print("변환된 값:", port)
print("유효한 포트 범위:", is_valid_range)

## 8. 임무 복기와 다음 자동화

다음을 코드와 말로 설명할 수 있는지 확인하세요.

- `=`와 `==`의 차이
- `443`과 `"443"`의 차이
- `bool("False")`가 `True`인 이유
- `None`과 `0`의 업무 의미 차이
- 형변환 성공과 포트 범위 유효성이 별도 문제인 이유

이번 실습에서 준비한 자료형은 이후 다음 자동화의 재료가 됩니다.

- 포트 스캐닝: 정수 포트 범위를 반복하고 응답 여부를 `bool`로 기록
- 브루트포스 탐지: 로그인 실패 횟수를 `int`로 집계하고 임계값을 판정
- DFIR: 여러 파일의 `bytes` 시그니처를 비교하고 미확인 정보를 `None`으로 보존

다음 절에서는 원시 문자열을 필드로 나누고, 이벤트 한 건을 `dict`, 여러 이벤트를 `list`로 구조화합니다. 그다음 조건문과 반복문으로 같은 판단을 여러 대상에 확장합니다.